In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

c:\Users\Nishant Varshney\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\Nishant Varshney\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model= ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:
class SentimentSchema(BaseModel):

    sentiment:Literal["positive","negative"]= Field(description='sentiment of the review')

In [19]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )

    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )

    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )

In [21]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [6]:
prompt = 'what is the sentiment of the review-the sentiment was to good'
structured_model.invoke(prompt).sentiment

'positive'

In [8]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis: dict
    response: str 

In [24]:
def find_sentiment(state:ReviewState):
    prompt = f'For thefollowing find the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return{'sentiment':sentiment}


def check_sentiment(state:ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state:ReviewState):
    prompt = f""" write a warm thank you message in respond to 
    this review: \n\n"{state['review']}\n
    also, kindly ask user to leave the feedback on our website
    """

    response= model.invoke(prompt).content
    return{"response":response}

def run_diagnosis(state:ReviewState):
    prompt = f"""diagnose this negative review:\n\n
    {state["review"]}\n
    Return issue_type,tone, and urgency. 
"""
    response=structured_model2.invoke(prompt)
    return{'diagnosis':response.model_dump()} #since pydantic/json format is thier so we need to convert it in dict

def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt= f""" you are a support assistant.
    the user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked 
    urgency as '{diagnosis['urgency']}'. write an empathetic,helpful resolution message.
    """
    response = model.invoke(prompt).content
    return{'response':response}

In [29]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)



workflow = graph.compile()

In [31]:
initial_state={
    'review':"I've been using this app for about a month now, and I must say, the user interface is incredibly clean and intuitive. Everything is exactly where you'd expect it to be. It's rare to find something that just works without needing a tutorial. Great job to the design team!"
}

workflow.invoke(initial_state)

{'review': "I've been using this app for about a month now, and I must say, the user interface is incredibly clean and intuitive. Everything is exactly where you'd expect it to be. It's rare to find something that just works without needing a tutorial. Great job to the design team!",
 'sentiment': 'positive',
 'response': 'Wow, thank you so much for this incredible review!\n\nWe\'re absolutely thrilled to hear that you\'ve been enjoying the app for the past month. Your comments about the user interface being "incredibly clean and intuitive" and how "everything is exactly where you\'d expect it to be" truly mean the world to us. It\'s our goal to create an experience that "just works" without the need for tutorials, so knowing we\'ve achieved that for you is fantastic.\n\nWe\'ll be sure to pass your kind words directly to our design team – they\'ll be delighted to hear their hard work is making such a positive impact!\n\nYour detailed feedback is incredibly valuable, and we\'d love to h